In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn
import numpy as np
import json
from dotenv import load_dotenv
from sklearn import KNearestClassifier

# pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)  
# pd.set_option('display.max_colwidth', None)
data = pd.read_json('appraisals_dataset.json')


**Get all subjects for each appraisal**

In [ ]:
all_subjects = []
for appraisal in data['appraisals']:
    all_subjects.append(appraisal['subject'])
    
df_subjects = pd.DataFrame(all_subjects)
df_subjects.info()

**Get all properties**

In [ ]:
all_properties = []
for appraisal in data['appraisals']:
    for property in appraisal['properties']:
        all_properties.append(property)

df_properties = pd.DataFrame(all_properties)
df_properties.info()


**Select numerical property features** 

In [ ]:
df_properties.describe()
df_subjects.describe()

In [ ]:
df_properties[['gla', 'room_count', 'full_baths', 'half_baths', 'bedrooms', 'lot_size_sf', 'year_built']].info()

In [ ]:
df_properties.isnull().sum()

**Drop property columns with too many missing data**

In [27]:
df_properties.drop(['main_level_finished_area', 'bg_fin_area', 'upper_lvl_fin_area', 'public_remarks'], axis=1, inplace=True)

**Select relevant columns common to both subject and properties**

In [45]:
print(f"unique columns in subjects: {df_subjects.columns.__len__()}")
print(f"unique columns in properties: {df_properties.columns.__len__()}")


unique columns in subjects: 35
unique columns in properties: 24


In [ ]:
props_selected = df_properties[['bedrooms', 'gla', 'year_built', 'structure_type', 'lot_size_sf', 'basement', 'heating', 'cooling', 'style']]
subs_selected = df_subjects[['num_beds', 'gla', 'year_built', 'structure_type', 'lot_size_sf', 'basement', 'heating','cooling', 'style']]

In [97]:
#set missing bedroom values with median
props_selected['bedrooms']
median = props_selected['bedrooms'].median()
props_selected.fillna({'bedrooms': median}, inplace=True)


C:\Users\dejhs\AppData\Local\Temp\ipykernel_36168\258159172.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  props_selected.fillna({'bedrooms': median}, inplace=True)


In [77]:
#convert number of beds for selected subjects to floats
subs_selected['num_beds'] = pd.to_numeric(subs_selected['num_beds'], errors='coerce')

C:\Users\dejhs\AppData\Local\Temp\ipykernel_36168\2518629405.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subs_selected['num_beds'] = pd.to_numeric(subs_selected['num_beds'], errors='coerce')


In [ ]:
#find out how many num_beds were not able to be converted
subs_selected['num_beds'].info()

#fill null values of num_beds with median
subs_selected['num_beds'] = subs_selected['num_beds'].fillna(subs_selected['num_beds'].median())


In [122]:
#clean gla values for subs_selected
import re
def clean_gla(gla_value):
    if isinstance(gla_value, str):
        # Remove commas
        gla_value = gla_value.replace(',', '')
        # Check for SqM or sqm (case-insensitive)
        if 'sqm' in gla_value.lower():
            num = float(re.sub('[^0-9.]+', '', gla_value.lower()))
            # 1 SqM = 10.7639 SqFt
            return num * 10.7639
        return float(re.sub('[^0-9.]+', '', gla_value.lower()))

subs_selected['gla'] = subs_selected['gla'].apply(clean_gla)

C:\Users\dejhs\AppData\Local\Temp\ipykernel_36168\3083706442.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subs_selected['gla'] = subs_selected['gla'].apply(clean_gla)


In [131]:
#clean gla values for props_selected(replacing missing values with median)
median = props_selected['gla'].median()
props_selected.fillna({'gla': median}, inplace=True)

C:\Users\dejhs\AppData\Local\Temp\ipykernel_36168\634465096.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  props_selected.fillna({'gla': median}, inplace=True)


In [ ]:
# Clean year_built values for subs_selected
# First convert to numeric, coercing errors to NaN
# subs_selected['year_built'] = pd.to_numeric(subs_selected['year_built'], errors='coerce')

# # Check the data
# print("Year Built Statistics:")
# print(subs_selected['year_built'].describe())
# print("\nMissing values:", subs_selected['year_built'].isnull().sum())

# # Fill missing values with median
# median_year = subs_selected['year_built'].median()
# subs_selected['year_built'] = subs_selected['year_built'].fillna(median_year)

# # Convert to integer since years should be whole numbers
# subs_selected['year_built'] = subs_selected['year_built'].astype(int)

# # Verify the cleaning
# print("\nAfter cleaning:")
# print(subs_selected['year_built'].describe())


Year Built Statistics:
count      79.000000
mean     1983.810127
std        39.757946
min      1845.000000
25%      1972.000000
50%      1998.000000
75%      2013.000000
max      2025.000000
Name: year_built, dtype: float64

Missing values: 9

After cleaning:
count      88.000000
mean     1985.261364
std        37.892921
min      1845.000000
25%      1974.750000
50%      1998.000000
75%      2010.250000
max      2025.000000
Name: year_built, dtype: float64


C:\Users\dejhs\AppData\Local\Temp\ipykernel_36168\3003610765.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subs_selected['year_built'] = pd.to_numeric(subs_selected['year_built'], errors='coerce')
C:\Users\dejhs\AppData\Local\Temp\ipykernel_36168\3003610765.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subs_selected['year_built'] = subs_selected['year_built'].fillna(median_year)
C:\Users\dejhs\AppData\Local\Temp\ipykernel_36168\3003610765.py:15: SettingWithCopyWarning: 
A value is trying to be 

In [ ]:
#drop year_built for both dataframes for now
subs_selected.drop(columns=['year_built'], inplace=True)
props_selected.drop(columns=['year_built'], inplace=True)


In [158]:
#clean lot_size_sf for both dataframes
props_selected.fillna({'lot_size_sf': props_selected['lot_size_sf'].median()}, inplace=True)


C:\Users\dejhs\AppData\Local\Temp\ipykernel_36168\2668174832.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  props_selected.fillna({'lot_size_sf': props_selected['lot_size_sf'].median()}, inplace=True)


<class 'pandas.core.series.Series'>
RangeIndex: 9820 entries, 0 to 9819
Series name: lot_size_sf
Non-Null Count  Dtype  
--------------  -----  
9820 non-null   float64
dtypes: float64(1)
memory usage: 76.8 KB
